In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

TAB_DIR = Path("../../tab/H7")


def chi2_v(ct):
    """Chi-kwadraat + Cramérs V, zonder Yates-correctie (V op onbewerkte chi2)."""
    ct = np.asarray(ct, dtype=float)
    chi2, p, _, exp = chi2_contingency(ct, correction=False)
    v = np.sqrt(chi2 / (ct.sum() * (min(ct.shape) - 1)))
    return chi2, p, v, exp

In [2]:
df = pd.read_excel("../../data/algemeen/gebeurtenisdata_meso/steekproefdatabank_gis.xlsx")
df.columns = df.columns.str.strip()
print(f"{len(df):,} rijen, {df.shape[1]} kolommen")

5,847 rijen, 42 kolommen


In [3]:
for kol in ["gender", "E_TIME_astro"]:
    df[kol] = df[kol].str.strip().str.lower()
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce")


def dag_of_nacht(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    if x == "dag":
        return "dag"
    if x == "nacht" or "schemer" in x:      # vangt schemer én schemering
        return "nacht"
    return None


df["dag_nacht"] = df["E_TIME_astro"].apply(dag_of_nacht)

# incident-id: incidentnummer volstaat binnen één steekproefbestand
uid = df["incidentnumber"].astype(str)
if "archive_file" in df.columns:
    uid = df["archive_file"].astype(str) + "_" + uid
df["incident_uid"] = uid

print("dag/nacht:", df["dag_nacht"].value_counts(dropna=False).to_dict())

dag/nacht: {'dag': 3501, 'nacht': 2324, None: 22}


In [4]:
# Groepsgrootte = aantal unieke personen per incident (elk gender telt mee).
# Groepstype  = op basis van de GEKENDE genders in het incident.
gedeeld = df.dropna(subset=["incident_uid"])
grootte = gedeeld.groupby("incident_uid")["ID"].nunique()

def bepaal_type(genders):
    known = [g for g in genders if g in ("man", "woman")]
    if not known:
        return "onbekend"
    heeft_m, heeft_v = "man" in known, "woman" in known
    if heeft_m and heeft_v:
        return "gemengd"
    return "exclusief vrouwelijk" if heeft_v else "exclusief mannelijk"

type_per_incident = (df.groupby("incident_uid")["gender"]
                       .apply(lambda s: bepaal_type(list(s))))

df["groepsgrootte"] = df["incident_uid"].map(grootte)
df["groepstype"] = np.where(df["groepsgrootte"] == 1, "alleen",
                            df["incident_uid"].map(type_per_incident))
df["sociaal"] = np.where(df["groepstype"] == "alleen", "alleen", "groep")

# Analyseset: man/vrouw, gekende dag/nacht
base = df[df["gender"].isin(["man", "woman"]) & df["dag_nacht"].notna()].copy()

# --- diagnostiek: welke definitie reproduceert Tabel 2 (doel groep-totaal = 370)? ---
jv = base[base["age"].between(15, 24)]
print("groep-totaal jongvolwassenen — op PERSOONSniveau:",
      (jv["sociaal"] == "groep").sum())
print("groep-totaal jongvolwassenen — op INCIDENTniveau:",
      jv[jv["sociaal"] == "groep"]
        .drop_duplicates(["incident_uid", "gender"]).shape[0])

groep-totaal jongvolwassenen — op PERSOONSniveau: 463
groep-totaal jongvolwassenen — op INCIDENTniveau: 355


In [5]:
jv = base[base["age"].between(15, 24)]

rijen = []
for periode in ["dag", "nacht"]:
    sub = jv[jv["dag_nacht"] == periode]
    ct = pd.crosstab(sub["gender"], sub["sociaal"])[["alleen", "groep"]]
    chi2, p, v, _ = chi2_v(ct)
    print(f"\n{periode.upper()}  (n={int(ct.values.sum())})")
    print(ct)
    print(f"chi2={chi2:.2f}, p={p:.3f}, Cramérs V={v:.3f}")

    rijen.append(ct.assign(periode=periode, chi2=chi2, p=p, V=v))

tabel_sociaal = pd.concat(rijen).reset_index().set_index(["periode", "gender"])
tabel_sociaal.to_csv(TAB_DIR / "7.2_jongvolwassenen_sociaal_dag_nacht.csv")


DAG  (n=605)
sociaal  alleen  groep
gender                
man         211    181
woman       113    100
chi2=0.03, p=0.855, Cramérs V=0.007

NACHT  (n=395)
sociaal  alleen  groep
gender                
man         173    130
woman        40     52
chi2=5.27, p=0.022, Cramérs V=0.115


In [6]:
groepen = jv[(jv["groepstype"] != "alleen") &
             (jv["groepstype"] != "onbekend")].copy()
groepen["samenstelling"] = np.where(groepen["groepstype"] == "gemengd",
                                    "heterogeen", "homogeen")

rijen = []
for g, label in [("woman", "VROUWEN"), ("man", "MANNEN")]:
    sub = groepen[groepen["gender"] == g]
    ct = pd.crosstab(sub["dag_nacht"], sub["samenstelling"])[["homogeen", "heterogeen"]]
    chi2, p, v, _ = chi2_v(ct)
    print(f"\n{label}  (n={int(ct.values.sum())})")
    print(ct)
    print(f"chi2={chi2:.2f}, p={p:.3f}, Cramérs V={v:.3f}")

    rijen.append(ct.assign(gender=g, chi2=chi2, p=p, V=v))

tabel_samenstelling = pd.concat(rijen).reset_index().set_index(["gender", "dag_nacht"])
tabel_samenstelling.to_csv(Path(TAB_DIR) / "7.3_jongvolwassenen_samenstelling_dag_nacht.csv")


VROUWEN  (n=152)
samenstelling  homogeen  heterogeen
dag_nacht                          
dag                  67          33
nacht                25          27
chi2=5.13, p=0.024, Cramérs V=0.184

MANNEN  (n=311)
samenstelling  homogeen  heterogeen
dag_nacht                          
dag                 147          34
nacht               105          25
chi2=0.01, p=0.921, Cramérs V=0.006


In [7]:
beroep = base[(base["gender"] == "woman") & base["prof_hisco"].notna()].copy()
beroep["beroep_cat"] = np.where(beroep["prof_hisco"] == "Prostitute",
                                "sekswerker", "overig")

# 1) Overall (2x3, incl. alleen)
ct_all = pd.crosstab(beroep["beroep_cat"], beroep["groepstype"])[
    ["alleen", "exclusief vrouwelijk", "gemengd"]]
chi2, p, v, _ = chi2_v(ct_all)
print(f"OVERALL (n={int(ct_all.values.sum())}): "
      f"chi2={chi2:.2f}, p={p:.4f}, Cramérs V={v:.3f}")
print(ct_all, "\n")

# 2) Per periode, ENKEL groepen (2x2: exclusief vrouwelijk vs gemengd)
for periode in ["dag", "nacht"]:
    sub = beroep[(beroep["dag_nacht"] == periode) &
                 (beroep["groepstype"].isin(["exclusief vrouwelijk", "gemengd"]))]
    ct = pd.crosstab(sub["beroep_cat"], sub["groepstype"])[
        ["exclusief vrouwelijk", "gemengd"]]
    n = int(ct.values.sum())
    _, _, _, exp = chi2_v(ct)
    if exp.min() < 5:                      # kleine cellen -> Fisher
        _, pf = fisher_exact(ct)
        print(f"{periode.upper()} (n={n}): Fishers exacte toets, p={pf:.3f}  "
              f"(chi2 onbetrouwbaar, kleinste verwachte cel {exp.min():.1f})")
    else:
        chi2, p, v, _ = chi2_v(ct)
        print(f"{periode.upper()} (n={n}): chi2={chi2:.2f}, p={p:.4f}, Cramérs V={v:.3f}")
    print(ct, "\n")

OVERALL (n=798): chi2=38.34, p=0.0000, Cramérs V=0.219
groepstype  alleen  exclusief vrouwelijk  gemengd
beroep_cat                                       
overig         372                   136      101
sekswerker      94                    83       12 

DAG (n=182): Fishers exacte toets, p=0.343  (chi2 onbetrouwbaar, kleinste verwachte cel 3.8)
groepstype  exclusief vrouwelijk  gemengd
beroep_cat                               
overig                       114       56
sekswerker                    10        2 

NACHT (n=150): chi2=48.50, p=0.0000, Cramérs V=0.569
groepstype  exclusief vrouwelijk  gemengd
beroep_cat                               
overig                        22       45
sekswerker                    73       10 

